# Phase 4D — 30-second live search demo

**Instructor requirement:** demonstrate the fastest version **running in real time**, inside the 15-minute performance-story presentation.

This notebook loads the existing **20K real Qwen responses**, **MiniLM 384d embedding model**, **Phase 4A fitted SVD k=100**, and **20K×100 reduced index**. It does **not** retrain or regenerate documents.

**Presenter workflow:** Run setup cells before presenting. During the 30-second demo, type a new English query into the interactive box, click Search, and show the real top-10 responses and measured latency.

**Timing:** Encoding time, SVD transform time, search+ranking time, and total query time are shown separately. Phase 4C's 100-query batch median is **not** single-query latency. The search method here is SVD reduced-index BLAS, matching Phase 4C's fastest 20K batch configuration; single-query speed is measured independently.

**Quality disclosure:** Phase 4B measured 77% average Top-10 overlap for 10 external queries using SVD k=100; overlap is not relevance accuracy. For this demo the top-10 results are approximate relative to full 384d cosine.

In [ ]:
%pip -q install sentence-transformers scikit-learn joblib ipywidgets threadpoolctl
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json,time,html,os
import numpy as np
import joblib
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits
from sentence_transformers import SentenceTransformer
from IPython.display import display,HTML,clear_output
import ipywidgets as widgets

ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
CORPUS=ROOT/'data/raw/qwen_corpus_20k_progress.jsonl'
MODEL=ROOT/'phase4_final_system/models/truncated_svd_k100_full_20k.joblib'
INDEX=ROOT/'phase4_final_system/models/reduced_index_k100_full_20k.npy'
for path in (CORPUS,MODEL,INDEX):
    assert path.exists(),f'Missing prerequisite: {path}'
print('All Phase 4D input files found.')

## 1. Load the prepared index and real source documents

This step can take time, so do it **before** the presentation. It is not part of the live 30 seconds.

In [ ]:
with open(CORPUS,'r',encoding='utf-8') as f:
    docs=[json.loads(line) for line in f if line.strip()]
assert len(docs)==20000,f'Expected 20K documents; got {len(docs)}'
svd=joblib.load(MODEL)
index=np.ascontiguousarray(np.load(INDEX),dtype=np.float32)
assert index.shape==(20000,100)
norms=np.linalg.norm(index,axis=1)
assert np.allclose(norms,1.0,atol=1e-4),'Expected pre-normalized SVD index; check Phase 4A.'
encoder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device='cpu')
print(f'READY: {len(docs):,} real documents | {index.shape[1]}d SVD index | MiniLM CPU encoder')
print('First document fields:',list(docs[0].keys()))

## 2. One real search, with end-to-end timing

A query is encoded with the **same MiniLM model** used for the corpus. We transform its 384d embedding with the **saved SVD model**, normalize it, compute cosine similarities against all 20K rows, then return the highest-scoring ten documents. We do **not** include data loading or model startup in per-query latency.

The display shows a short excerpt of each actual Qwen response, plus document ID and cosine score. We show *separate* encoding and search times so the instructor can distinguish user-visible response time from benchmarked vector-search time.

In [ ]:
def run_search(query,top_k=10):
    if not query or not query.strip():raise ValueError('Enter a nonempty English query.')
    start=time.perf_counter()
    vector=encoder.encode([query.strip()],convert_to_numpy=True,normalize_embeddings=False,show_progress_bar=False)
    after_encoding=time.perf_counter()
    reduced=svd.transform(np.asarray(vector,dtype=np.float32))
    reduced=normalize(reduced,axis=1).astype(np.float32,copy=False)
    after_projection=time.perf_counter()
    with threadpool_limits(limits=1):
        scores=index@reduced[0]
        # Identical stable tie-break rule as Phase 4C.
        best=np.lexsort((np.arange(len(scores)),-scores))[:top_k]
    after_search=time.perf_counter()
    results=[]
    for i in best:
        d=docs[int(i)]
        results.append({'row':int(i),'id':str(d.get('id',i)),
                        'score':float(scores[i]),
                        'prompt':str(d.get('prompt','')),
                        'response':str(d.get('response',d.get('text','')))})
    return {'query':query,'results':results,
            'encoding_ms':(after_encoding-start)*1000,
            'svd_ms':(after_projection-after_encoding)*1000,
            'search_ms':(after_search-after_projection)*1000,
            'total_ms':(after_search-start)*1000}

# Pre-warm the encoder, SVD transform, BLAS, and ranking before presentation.
warmup=run_search('How can machine learning help detect cybersecurity threats?',top_k=3)
print('Warmup passed. Search latency (warmup, not benchmark):',round(warmup['total_ms'],2),'ms')
print('Ready for interactive demonstration.')

## 3. LIVE DEMO — only this widget is needed on screen

**30-second presentation:** Enter a new English query, press Search, show actual top-10 results and real measured latency. The query is not hardcoded and can be chosen by the instructor.

**Optional rehearsal:** Run the example once, then replace it with another query during the presentation. If Colab disconnects, reconnect and run setup cells again; screenshots/videos are only a backup, not a substitute for the required live demo.

In [ ]:
query_box=widgets.Text(value='How can AI help detect cyber attacks?',
    placeholder='Type a question in English',description='Query:',layout=widgets.Layout(width='90%'))
search_button=widgets.Button(description='Search top 10',button_style='success',
    icon='search',layout=widgets.Layout(width='180px'))
status=widgets.HTML(value='<span style="color:#0f766e">● Ready — 20,000 documents loaded</span>')
results_box=widgets.Output(layout=widgets.Layout(border='1px solid #ddd',padding='12px'))

def handle_search(_):
    search_button.disabled=True
    status.value='<span>Searching…</span>'
    with results_box:
        clear_output(wait=True)
        try:
            result=run_search(query_box.value,10)
            title=html.escape(result['query'])
            print('QUERY:',result['query'])
            print(f"Total: {result['total_ms']:.2f} ms | MiniLM encoding: {result['encoding_ms']:.2f} ms | SVD transform: {result['svd_ms']:.2f} ms | Vector search + top-10 ranking: {result['search_ms']:.2f} ms")
            print('-'*75)
            for rank,r in enumerate(result['results'],1):
                excerpt=' '.join(r['response'].split())
                if not excerpt:excerpt='[Empty response — inspect source JSONL]'
                if len(excerpt)>230:excerpt=excerpt[:227]+'...'
                print(f"{rank:2}. doc={r['id']} | cosine={r['score']:.4f}")
                print('    '+excerpt)
                print()
            status.value='<span style="color:#0f766e">● Search complete — real results</span>'
        except Exception as exc:
            print(type(exc).__name__+':',exc)
            status.value='<span style="color:#b91c1c">● Search failed — inspect output</span>'
        finally:
            search_button.disabled=False
search_button.on_click(handle_search)
display(widgets.VBox([widgets.HTML('<h3>HP-SS Engine | Live Semantic Search (20K documents)</h3>'),
                      query_box,search_button,status,results_box]))
print('TIP: Press the green Search top 10 button to run a REAL search.')

## Instructor presentation notes (not part of the 30 seconds)

- **Problem:** find top-10 semantically similar texts from 20K real Qwen responses.
- **Baseline:** Phase 1 pure Python cosine; be explicit that its TF-IDF workload differs from MiniLM and use Phase 3D same-vector comparison for fair speedup claims.
- **Journey:** vectorization and BLAS, batching, SVD dimensionality reduction, parallelism comparisons.
- **Result:** Phase 4C full batch median at 20K was 190.79 ms for 100 queries using SVD batched 100d; **the single live query has its own measured time**, displayed above. Phase 4B external-query SVD Top-10 overlap was 77%.
- **Lessons:** discuss measured behavior and variability; don't claim a specific bottleneck without evidence.
- **10× data:** 200K×384 float32 ~293 MiB for raw embeddings alone; extrapolation is not a measured latency.
- **Live demo:** run setup in advance; during the demo only interact with the final search widget. Aim for ~30 seconds of presentation time.
- **Reliability:** if results look irrelevant, investigate semantic model and dataset characteristics; do not handpick fake answers.

After pilot verification, archive notebook and final screenshot/results as evidence, then build the final presentation story and report.